#### **Newcastle Urban Heat Vulnerability Index (UHVI) Audience Adaptive GeoXAI Dashboard**

*Colab-native dashboard using plotly and ipywidgets based on GeoAI analysis:*

- **Resident mode:** gives a plain-language neighbourhood explanation.
- **Urban planner mode:** gives intervention benefit and priority.
- **Public health mode:** gives exposure–sensitivity–adaptive-capacity decomposition.
- **Researcher mode:** gives SHAP explanation and model-performance framing.


In [19]:
# Mount Google Drive

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [20]:
# Install and import

!pip install geopandas plotly ipywidgets pyogrio -q

import os, json
import numpy as np
import pandas as pd
import geopandas as gpd
import plotly.express as px
import ipywidgets as widgets
from IPython.display import display


In [21]:
# Load data

base_dir = "/content/drive/MyDrive/LST_Project"
output_dir = f"{base_dir}/outputs"

possible_files = [
    f"{output_dir}/newcastle_geoxai_adaptation_scenarios.gpkg",
    f"{output_dir}/newcastle_uhvi_shap_values.gpkg",
    f"{output_dir}/newcastle_full_uhvi.gpkg"
]

data_path = next((p for p in possible_files if os.path.exists(p)), None)

if data_path is None:
    raise FileNotFoundError("No GeoXAI/UHVI file found.")

print("Using:", data_path)

gdf = gpd.read_file(data_path).to_crs("EPSG:4326").reset_index(drop=True)
gdf["map_id"] = gdf.index.astype(str)

name_col = "LSOA21NM_x" if "LSOA21NM_x" in gdf.columns else (
    "LSOA21NM" if "LSOA21NM" in gdf.columns else "LSOA21CD"
)

shap_cols = [c for c in gdf.columns if c.startswith("shap_")]

if "top_driver" not in gdf.columns and shap_cols:
    gdf["top_driver"] = (
        gdf[shap_cols].abs().idxmax(axis=1).str.replace("shap_", "", regex=False)
    )

if "top_driver" not in gdf.columns:
    gdf["top_driver"] = "Not available"

Using: /content/drive/MyDrive/LST_Project/outputs/newcastle_geoxai_adaptation_scenarios.gpkg


In [22]:
# Recalculate missing sub-indices

def minmax(series):
    series = pd.to_numeric(series, errors="coerce")
    if series.max() == series.min():
        return series * 0
    return (series - series.min()) / (series.max() - series.min())

exposure_vars = [
    "lst_mean",
    "lst_p95",
    "hotspot_binary"
]

sensitivity_vars = [
    "age_65_pct",
    "age_75_pct",
    "disability_pct",
    "health_deprivation_score",
    "population_density"
]

adaptive_vars = [
    "tree_canopy_pct",
    "green_pct",
    "income_score"
]

for col in exposure_vars + sensitivity_vars + adaptive_vars:
    if col not in gdf.columns:
        gdf[col] = np.nan
    gdf[col + "_norm"] = minmax(gdf[col])

gdf["exposure_index"] = gdf[[c + "_norm" for c in exposure_vars]].mean(axis=1)
gdf["sensitivity_index"] = gdf[[c + "_norm" for c in sensitivity_vars]].mean(axis=1)
gdf["adaptive_capacity_index"] = gdf[[c + "_norm" for c in adaptive_vars]].mean(axis=1)

if "uhvi_class" not in gdf.columns:
    gdf["uhvi_class"] = pd.qcut(
        gdf["uhvi"],
        q=5,
        labels=["Very Low", "Low", "Moderate", "High", "Very High"]
    )


In [23]:
# Recalculate scenario benefit if missing

scenario_cols = [
    "uhvi_reduction_tree_20",
    "uhvi_reduction_green_20",
    "uhvi_reduction_impervious_10",
    "uhvi_reduction_combined"
]

for col in scenario_cols:
    if col not in gdf.columns:
        gdf[col] = np.nan

# If no scenario columns exist, create proxy benefits from relevant indicators
if gdf[scenario_cols].isna().all().all():
    gdf["uhvi_reduction_tree_20"] = minmax(100 - gdf["tree_canopy_pct"]) * gdf["uhvi"] * 0.15
    gdf["uhvi_reduction_green_20"] = minmax(100 - gdf["green_pct"]) * gdf["uhvi"] * 0.12
    gdf["uhvi_reduction_impervious_10"] = minmax(gdf.get("impervious_pct", gdf["lst_mean"])) * gdf["uhvi"] * 0.08
    gdf["uhvi_reduction_combined"] = (
        gdf["uhvi_reduction_tree_20"]
        + gdf["uhvi_reduction_green_20"]
        + gdf["uhvi_reduction_impervious_10"]
    )

# Priority = high vulnerability + high combined benefit
gdf["intervention_priority"] = (
    minmax(gdf["uhvi"]) + minmax(gdf["uhvi_reduction_combined"])
) / 2

gdf["priority_class"] = pd.qcut(
    gdf["intervention_priority"],
    q=5,
    labels=["Very Low", "Low", "Moderate", "High", "Very High"]
)

geojson = json.loads(gdf.to_json())


In [24]:
# Dynamic explanation functions

def level(value, series):
    q1 = series.quantile(0.33)
    q2 = series.quantile(0.66)
    if value <= q1:
        return "low"
    elif value <= q2:
        return "moderate"
    return "high"

def resident_explanation(row):
    reasons = []

    if level(row["lst_mean"], gdf["lst_mean"]) == "high":
        reasons.append("high surface temperature")

    if level(row["tree_canopy_pct"], gdf["tree_canopy_pct"]) == "low":
        reasons.append("low tree canopy")

    if level(row["green_pct"], gdf["green_pct"]) == "low":
        reasons.append("limited greenspace")

    if "impervious_pct" in gdf.columns and level(row["impervious_pct"], gdf["impervious_pct"]) == "high":
        reasons.append("many hard or paved surfaces")

    if len(reasons) == 0:
        reasons.append("a combination of moderate heat exposure and limited cooling capacity")

    message = "This area is vulnerable mainly because of " + ", ".join(reasons) + "."

    if row["tree_canopy_pct"] < gdf["tree_canopy_pct"].median():
        advice = "Increasing tree canopy would be a practical local cooling option."
    elif row["green_pct"] < gdf["green_pct"].median():
        advice = "Improving greenspace could help reduce local heat risk."
    else:
        advice = "Maintaining existing vegetation is important for keeping heat risk lower."

    return message, advice

def planner_explanation(row, scenario_col):
    benefit = row[scenario_col]

    if row["uhvi"] >= gdf["uhvi"].quantile(0.8) and benefit >= gdf[scenario_col].quantile(0.8):
        return "High priority: this LSOA is both highly vulnerable and likely to benefit strongly from intervention."
    elif row["uhvi"] >= gdf["uhvi"].quantile(0.8):
        return "High vulnerability, but estimated intervention benefit is moderate. Further local design assessment is needed."
    elif benefit >= gdf[scenario_col].quantile(0.8):
        return "Good intervention opportunity, but not currently among the most vulnerable LSOAs."
    else:
        return "Lower immediate priority compared with other Newcastle LSOAs."

def health_explanation(row):
    e = row["exposure_index"]
    s = row["sensitivity_index"]
    a = row["adaptive_capacity_index"]

    if e > s and e > a:
        return "Heat exposure is the dominant concern in this LSOA."
    elif s > e and s > a:
        return "Population sensitivity is the dominant concern in this LSOA."
    elif a < 0.33:
        return "Low adaptive capacity is a key concern, suggesting limited cooling or social resilience."
    else:
        return "This LSOA has a mixed vulnerability profile."

def researcher_explanation(row):
    if not shap_cols:
        return "SHAP values are not available for this dataset."

    shap_values = row[shap_cols].sort_values(key=lambda x: x.abs(), ascending=False)
    top = shap_values.index[0].replace("shap_", "")
    val = shap_values.iloc[0]

    direction = "increases" if val > 0 else "reduces"

    return f"For this LSOA, {top} is the strongest local SHAP driver and it {direction} predicted UHVI by {val:.3f}."

In [25]:
# Widgets

audience_widget = widgets.Dropdown(
    options=["Resident", "Urban planner", "Public health", "Researcher"],
    value="Resident",
    description="Audience:",
    layout=widgets.Layout(width="320px")
)

scenario_widget = widgets.Dropdown(
    options=[
        ("Tree canopy +20%", "uhvi_reduction_tree_20"),
        ("Greenspace +20%", "uhvi_reduction_green_20"),
        ("Impervious surface -10%", "uhvi_reduction_impervious_10"),
        ("Combined", "uhvi_reduction_combined"),
    ],
    value="uhvi_reduction_combined",
    description="Scenario:",
    layout=widgets.Layout(width="360px")
)

lsoa_widget = widgets.Dropdown(
    options=[
        (f"{r[name_col]} | {r['LSOA21CD']} | UHVI {r['uhvi']:.3f}", r["map_id"])
        for _, r in gdf.sort_values("uhvi", ascending=False).iterrows()
    ],
    value=gdf.sort_values("uhvi", ascending=False).iloc[0]["map_id"],
    description="LSOA:",
    layout=widgets.Layout(width="780px")
)

panel_output = widgets.Output()

In [26]:
# Render dashboard

def render_dashboard(*args):
    audience = audience_widget.value
    selected_id = lsoa_widget.value
    scenario_col = scenario_widget.value

    row = gdf[gdf["map_id"] == selected_id].iloc[0]

    with panel_output:
        panel_output.clear_output(wait=True)

        print(f"Selected LSOA: {row[name_col]}")
        print(f"LSOA code: {row['LSOA21CD']}")
        print(f"UHVI: {row['uhvi']:.3f}")
        print(f"UHVI class: {row['uhvi_class']}")
        print()

        if audience == "Resident":
            reason, advice = resident_explanation(row)
            print("Resident-friendly explanation")
            print("-----------------------------")
            print("Why is this area vulnerable?")
            print(reason)
            print()
            print("Plain-language message")
            print(advice)

        elif audience == "Urban planner":
            print("Urban planner intervention view")
            print("-------------------------------")
            print(f"Selected scenario: {scenario_col}")
            print(f"Estimated benefit: {row[scenario_col]:.3f}")
            print(f"Priority class: {row['priority_class']}")
            print()
            print(planner_explanation(row, scenario_col))

        elif audience == "Public health":
            print("Public-health vulnerability decomposition")
            print("----------------------------------------")
            print(f"Exposure index: {row['exposure_index']:.3f}")
            print(f"Sensitivity index: {row['sensitivity_index']:.3f}")
            print(f"Adaptive capacity index: {row['adaptive_capacity_index']:.3f}")
            print()
            print(f"Population density: {row['population_density']:.2f}")
            print(f"Age 65+: {row['age_65_pct']:.2f}%")
            print(f"Disability: {row['disability_pct']:.2f}%")
            print(f"Health deprivation: {row['health_deprivation_score']:.2f}")
            print()
            print(health_explanation(row))

        elif audience == "Researcher":
            print("Researcher-oriented GeoXAI explanation")
            print("-------------------------------------")
            print(f"Top driver: {row['top_driver']}")
            print(researcher_explanation(row))
            print()
            print("Project-level model performance: R² ≈ 0.960, mean CV R² ≈ 0.945.")

In [27]:
# Display

audience_widget.observe(render_dashboard, names="value")
scenario_widget.observe(render_dashboard, names="value")
lsoa_widget.observe(render_dashboard, names="value")

dashboard = widgets.VBox([
    widgets.HTML("<h2>Newcastle Audience-Adaptive GeoXAI Dashboard</h2>"),

    # Map slicer removed
    widgets.HBox([
        audience_widget,
        scenario_widget,
        lsoa_widget
    ]),

    panel_output
])

display(dashboard)

render_dashboard()